In [ ]:
# !pip install pandas matplotlib requests 

import requests
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from io import StringIO

# Setting visual style
plt.style.use('fivethirtyeight')
%matplotlib inline

In [ ]:
endpoint_url = "https://query.wikidata.org/sparql"

sparql_query = """
SELECT ?item ?itemLabel ?relatedItem ?relatedItemLabel ?pLabel WHERE {
  BIND(wd:Q510144 AS ?museum)
  
  {
    # Level 1: Museum to Exhibitions
    ?exhibition wdt:P31/wdt:P279* wd:Q464980; 
                wdt:P276 ?museum.
    BIND(?museum AS ?item)
    BIND(?exhibition AS ?relatedItem)
    BIND("location" AS ?pLabel)
  }
  UNION
  {
    # Level 2: Exhibitions to Artists/Curators/Subjects
    ?exhibition wdt:P31/wdt:P279* wd:Q464980; 
                wdt:P276 wd:Q510144;
                ?p ?relatedItem.
    
    FILTER(?p IN (wdt:P170, wdt:P1640, wdt:P921, wdt:P180)) 
    BIND(?exhibition AS ?item)
    
    SERVICE wikibase:label { 
      bd:serviceParam wikibase:language "[AUTO_LANGUAGE],en,de". 
      ?p wikibase:propertyLabel ?pLabel.
    }
  }
  SERVICE wikibase:label { 
    bd:serviceParam wikibase:language "[AUTO_LANGUAGE],en,de". 
    ?item rdfs:label ?itemLabel.
    ?relatedItem rdfs:label ?relatedItemLabel.
  }
}
LIMIT 500
"""

In [ ]:
headers = {'User-Agent': 'SprengelInventoryBot/1.0', 'Accept': 'application/sparql-results+json'}
response = requests.get(endpoint_url, params={'query': sparql_query, 'format': 'json'}, headers=headers)

if response.status_code == 200:
    data = response.json()
    results = []
    for row in data['results']['bindings']:
        results.append({
            'Subject': row['itemLabel']['value'],
            'Object': row['relatedItemLabel']['value'],
            'Relationship': row['pLabel']['value']
        })
    df = pd.DataFrame(results)
    print(f"Successfully retrieved {len(df)} records.")
else:
    print(f"Error: {response.status_code}")

In [ ]:
# Data Aggregation
total_exhibitions = df[df['Relationship'] == 'location']['Object'].nunique()
total_participants = df[df['Relationship'] != 'location']['Object'].nunique()
property_counts = df[df['Relationship'] != 'location']['Relationship'].value_counts()

# Plotting
fig = plt.figure(figsize=(15, 8))

# 1. KPI Panel
ax1 = plt.subplot2grid((2, 2), (0, 0), colspan=2)
ax1.axis('off')
ax1.text(0.2, 0.6, f"{total_exhibitions}", fontsize=60, fontweight='bold', color='#841617', ha='center')
ax1.text(0.2, 0.3, "Catalogued Exhibitions", fontsize=14, ha='center')

ax1.text(0.5, 0.6, f"{total_participants}", fontsize=60, fontweight='bold', color='#2C3E50', ha='center')
ax1.text(0.5, 0.3, "Unique Participants", fontsize=14, ha='center')

ax1.text(0.8, 0.6, f"{len(df)}", fontsize=60, fontweight='bold', color='#7F8C8D', ha='center')
ax1.text(0.8, 0.3, "Total Statements", fontsize=14, ha='center')

# 2. Relationship Breakdown
ax2 = plt.subplot2grid((2, 2), (1, 0))
property_counts.plot(kind='barh', ax=ax2, color=['#841617', '#A04040', '#4D4D4D'])
ax2.set_title("Statements by Property Type")

# 3. Data Sample Table
ax3 = plt.subplot2grid((2, 2), (1, 1))
ax3.axis('off')
table_data = df.head(10)
ax3.table(cellText=table_data.values, colLabels=table_data.columns, loc='center', cellLoc='left')
ax3.set_title("Inventory Sample (First 10 Rows)")

plt.tight_layout()
plt.show()

In [ ]:
df.to_csv('sprengel_museum_inventory.csv', index=False)
print("Inventory exported to 'sprengel_museum_inventory.csv'")